# HVAC Electircity Demand Analysis and Prediction

## Feature Creation and lag features


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor

from scripts.notebook_utils import (
    add_lag_features,
    add_shifted_rolling_features,
    correlation_threshold_time_series_train,
    evaluate_regression,
    mape,
    select_correlated_features,
    split_dataframe_chronologically,
)

np.random.seed(42)


In [ ]:
df = pd.read_csv("./data/Load_data_01.csv")
df["Time"] = pd.to_datetime(df["Time"])
df.set_index("Time", inplace=True)

In [ ]:
df_daily_rec = pd.read_csv("./data/df_daily_feature_creation.csv")
df_daily_rec["Time"] = pd.to_datetime(df_daily_rec["Time"])
df_daily_rec.set_index("Time", inplace=True)

In [ ]:
df_daily = df_daily_rec

In [ ]:
df_daily

In [ ]:
df_daily.describe().T

## Sliding window method for electricity demand and heating demand


### Add lag features for electricity demand and heating demand (sliding window method)

- Average electricity demand in the past few days
- Average heating demand standard deviation in the past few days
- Electricity demand per day in the past few days


In [ ]:
df_daily["heat_demand_values"] = df["heat_demand_values"].resample("D").sum()

In [ ]:
window_sizes = [7, 14]
lag_days = [1, 6, 13]

df_daily = add_shifted_rolling_features(
    df_daily,
    target_columns=["electricity_demand_values", "heat_demand_values"],
    window_sizes=window_sizes,
    shift=1,
)
df_daily = add_lag_features(
    df_daily,
    "electricity_demand_values",
    lag_days,
    prefix="electricity_demand_lag",
)


In [ ]:
df_daily.dropna(inplace=True)

In [ ]:
df_daily

## Training and Testing models


In [ ]:
def train_test_set(df, start, end, split_time):
    train = df[(df.index > start) & (df.index <= split_time)]
    test = df[(df.index > split_time) & (df.index <= end)]
    X_train, y_train = (
        train.drop(["electricity_demand_values"], axis=1),
        train["electricity_demand_values"],
    )
    X_test, y_test = (
        test.drop(["electricity_demand_values"], axis=1),
        test["electricity_demand_values"],
    )
    return X_train, y_train, X_test, y_test

In [ ]:
X_train, y_train, X_test, y_test = train_test_set(
    df_daily, "2017-01-01", "2018-11-20", "2018-06-01"
)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
_, ax = plt.subplots(figsize=(12, 3))

ax.plot(y_train.index, y_train, label="Train")
ax.plot(y_test.index, y_test, label="Test")

plt.legend()
plt.tight_layout()

plt.show()

In [ ]:
def MAPE(y_true, y_pred):  # Calculate Mean Absolute Percentage Error
    return mape(y_true, y_pred)


In [ ]:
def evaluate_model_performance(
    y_true, y_pred
):  # def the output function for model performance evaluation
    return evaluate_regression(y_true, y_pred)


In [ ]:
def train_model(X_train, y_train, X_test, y_test, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    rmse, mae, mape, r2 = evaluate_model_performance(y_test, y_pred)
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MAPE: {mape:.2f} %")
    print(f"R2: {r2:.4f}")
    df_performance = pd.DataFrame(
        {"RMSE": rmse, "MAE": mae, "MAPE(%)": mape, f"$R^2$": r2}, index=[0]
    )
    df_results = pd.DataFrame(
        {"Time": y_test.index, "y_test": y_test, "y_pred": y_pred}
    )
    return df_performance, df_results

In [ ]:
def plot_results(df_results, model_name):
    _, ax = plt.subplots(figsize=(12, 3))
    ax.plot(df_results["Time"], df_results["y_test"], label="Test")
    ax.plot(df_results["Time"], df_results["y_pred"], label="Predicted")
    ax.set_title(model_name)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
dt_reg = DecisionTreeRegressor(max_depth=10, random_state=42)
df_performance, df_results = train_model(X_train, y_train, X_test, y_test, dt_reg)

In [ ]:
df_performance

In [ ]:
plot_results(df_results, "Decision Tree Regressor")

In [ ]:
RF_reg = RandomForestRegressor(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
)

df_performance, df_results = train_model(X_train, y_train, X_test, y_test, RF_reg)

In [ ]:
df_performance

In [ ]:
plot_results(df_results, "Random Forest Regressor")

In [ ]:
ada_reg = AdaBoostRegressor(
    DecisionTreeRegressor(max_depth=10, random_state=42),
    n_estimators=100,
    random_state=42,
)

df_performance, df_results = train_model(X_train, y_train, X_test, y_test, ada_reg)

In [ ]:
df_performance

In [ ]:
plot_results(df_results, "AdaBoost Regressor with Decision Tree Regressor")

In [ ]:
_, ax = plt.subplots(figsize=(16, 16))
sns.heatmap(df_daily.corr(), annot=True, ax=ax)
ax.set_title("Correlation Matrix of Daily Data with created features")
plt.tight_layout()
plt.show()

## Features selection


### Remove meterological features


In [ ]:
df_daily.drop(
    columns=[
        "air_pressure",
        "solar_irridiation_positive",
        "total_cloud_cover_percent",
        "wind_speed_range",
        "Wind scale 2",
        "Wind scale 3",
        "huimidity_Comfort",
        "huimidity_Uncomfortable Dry",
        "huimidity_Uncomfortable Wet",
        "temp_0_10",
        "temp_10_20",
        "temp_20_30",
        "temp_30_40",
        "temp_minus_10_0",
    ],
    inplace=True,
)

In [ ]:
X_train, y_train, X_test, y_test = train_test_set(
    df_daily, "2017-01-01", "2018-11-20", "2018-06-01"
)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
dt_reg = DecisionTreeRegressor(max_depth=10, random_state=42)
RF_reg = RandomForestRegressor(
    n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
)
ada_reg = AdaBoostRegressor(
    DecisionTreeRegressor(max_depth=10, random_state=42),
    n_estimators=100,
    random_state=42,
)

In [ ]:
reg_ls = [dt_reg, RF_reg, ada_reg]
reg_ls_name = ["Decision Tree", "Random Forest", "AdaBoost"]


def model_evaluation(reg_ls, reg_ls_name, X_train, y_train, X_test, y_test):
    performance_results = {}
    for reg, reg_name in zip(reg_ls, reg_ls_name):
        df_performance = train_model(X_train, y_train, X_test, y_test, reg)[0]
        performance_results[reg_name] = df_performance

    performance_results_df = (
        pd.concat(performance_results, axis=0)
        .reset_index(level=0)
        .rename(columns={"level_0": "Model Name"})
    )

    return performance_results_df

In [ ]:
performance_results_df = model_evaluation(
    reg_ls, reg_ls_name, X_train, y_train, X_test, y_test
)

In [ ]:
performance_results_df

In [ ]:
_, ax = plt.subplots(figsize=(16, 16))
sns.heatmap(df_daily.corr(), annot=True, ax=ax)
ax.set_title("Correlation Matrix of Daily Data with created features")
plt.tight_layout()
plt.show()

### Remove time features


In [ ]:
df_daily.drop(
    columns=[
        "Is_Weekday_0",
        "Is_Weekday_1",
        "Is_Weekday_2",
        "Is_Weekday_3",
        "Is_Weekday_4",
        "Is_Weekday_5",
        "Is_Weekday_6",
        "month_1",
        "month_2",
        "month_3",
        "month_4",
        "month_5",
        "month_6",
        "month_7",
        "month_8",
        "month_9",
        "month_10",
        "month_11",
        "month_12",
    ],
    inplace=True,
)

In [ ]:
X_train, y_train, X_test, y_test = train_test_set(
    df_daily, "2017-01-01", "2018-11-20", "2018-06-01"
)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
performance_results_df = model_evaluation(
    reg_ls, reg_ls_name, X_train, y_train, X_test, y_test
)

In [ ]:
performance_results_df

In [ ]:
_, ax = plt.subplots(figsize=(16, 16))
sns.heatmap(df_daily.corr(), annot=True, ax=ax)
ax.set_title("Correlation Matrix of Daily Data with created features")
plt.tight_layout()
plt.show()

### Remove the rolling mean and standard deviation features

- Keep the lag features for electricity demand only


In [ ]:
df_daily = df_daily_rec[["electricity_demand_values"]]

In [ ]:
lag_days = [1, 6, 13]

df_daily = add_lag_features(
    df_daily,
    "electricity_demand_values",
    lag_days,
    prefix="electricity_demand_lag",
)
df_daily.dropna(inplace=True)


In [ ]:
X_train, y_train, X_test, y_test = train_test_set(
    df_daily, "2017-01-01", "2018-11-20", "2018-06-01"
)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
performance_results_df = model_evaluation(
    reg_ls, reg_ls_name, X_train, y_train, X_test, y_test
)

In [ ]:
performance_results_df

### Remove the lag features for electricity demand, keep rolling mean and standard deviation features


In [ ]:
df_daily = df_daily_rec[["electricity_demand_values"]]

In [ ]:
df_daily["heat_demand_values"] = df["heat_demand_values"].resample("D").sum()

window_sizes = [7, 14]
df_daily = add_shifted_rolling_features(
    df_daily,
    target_columns=["electricity_demand_values", "heat_demand_values"],
    window_sizes=window_sizes,
    shift=1,
)
df_daily.dropna(inplace=True)


In [ ]:
X_train, y_train, X_test, y_test = train_test_set(
    df_daily, "2017-01-01", "2018-11-20", "2018-06-01"
)
X_train.shape, y_train.shape, X_test.shape, y_test.shape

In [ ]:
performance_results_df = model_evaluation(
    reg_ls, reg_ls_name, X_train, y_train, X_test, y_test
)

In [ ]:
performance_results_df

- Performance is worse than the model with lag features for electricity demand


### Add more electricity demand lag features

- The feature selection result indicates electricity demand lag features' importace to the model's predictive capacity. Consider the electricity demand seasonality, we will examine the model's performance with more electricity demand lag features next.


In [ ]:
df_daily = df_daily_rec[["electricity_demand_values"]]

In [ ]:
lag_days = np.arange(1, 61, 1)

for lag in lag_days:
    df_daily[f"electricity_demand_lag_{lag}"] = df_daily[
        "electricity_demand_values"
    ].shift(lag)

In [ ]:
df_daily.dropna(inplace=True)

In [ ]:
def corr_features_plot(df_daily, threshold):
    correlations = df_daily.corr()["electricity_demand_values"]
    correlated_features = correlations[abs(correlations) > threshold].index
    _, ax = plt.subplots(figsize=(16, 16))
    sns.heatmap(
        df_daily.corr().loc[correlated_features, correlated_features], annot=True, ax=ax
    )
    ax.set_title(f"Correlation Matrix of Daily Data (abs(corr) >= {threshold})")
    plt.tight_layout()
    plt.show()

In [ ]:
corr_features_plot(df_daily, 0.3)

### Find the relationship between the correlation coefficient and the model's performance


In [ ]:
# Time-series validation is handled by shared helpers that calculate MAPE directly.


In [ ]:
def correlation_threshold_train(df_daily, threshold, model_name, regressor, split=5):
    return correlation_threshold_time_series_train(
        df_daily,
        target="electricity_demand_values",
        threshold=threshold,
        model_name=model_name,
        regressor=regressor,
        n_splits=split,
        test_size=0.2,
    )


In [ ]:
ada_reg = AdaBoostRegressor(
    DecisionTreeRegressor(max_depth=10, random_state=42),
    n_estimators=100,
    random_state=42,
)

In [ ]:
thresholds = [0, 0.1, 0.3, 0.5, 0.6, 0.7, 0.8]

results_ls = [
    correlation_threshold_train(df_daily, _, "AdaBoost", ada_reg, 5) for _ in thresholds
]

df_results = pd.DataFrame(results_ls)

In [ ]:
df_results

### Select features based on the correlation threshold 0.3


In [ ]:
trainval_df, test_df = split_dataframe_chronologically(df_daily, test_size=0.2)
correlations, correlated_features = select_correlated_features(
    trainval_df, "electricity_demand_values", 0.3
)


In [ ]:
X_trainval = trainval_df[correlated_features].drop("electricity_demand_values", axis=1)
y_trainval = trainval_df["electricity_demand_values"]
X_test = test_df[correlated_features].drop("electricity_demand_values", axis=1)
y_test = test_df["electricity_demand_values"]


In [ ]:
print("AdaBoost Regressor with Decision Tree Regressor 5 Fold TimeSeriesSplit Results:\n")
fold_metrics_df, summary = time_series_cv_evaluation(
    ada_reg, X_trainval, y_trainval, n_splits=5
)
evaluation_df = pd.DataFrame([summary])
fold_metrics_df


In [ ]:
evaluation_df

In [ ]:
y_pred = ada_reg.predict(X_test)

In [ ]:
rmse, mae, mape_score, r2 = evaluate_regression(y_test, y_pred)
print(
    f"Training on holdout set: RMSE: {rmse:.4f}, MAE: {mae:.4f}, "
    f"MAPE:{mape_score:.4f} %, R2: {r2:.4f}"
)

df_results = pd.DataFrame({"y_test": y_test, "y_pred": y_pred})


In [ ]:
df_results.plot(figsize=(16, 6))

In [ ]:
len(df_results)

In [ ]:
test_idx = df_results.index.sort_values()[310:340]

In [ ]:
df_results.loc[test_idx].plot(figsize=(16, 6))

In [ ]:
# df_daily.to_csv("./data/df_daily_feature_lags.csv")

## Conclusion

- Lag features helped the model to predict electricity demand and heating demand better on all evaluation metrics.
- By correlation coefficient matrix, we can find the relationships between present electricity demand and the number of previous days’ demand. With high correlated lag features, the model can predict the future demand with high accuracy as mentioned above.

